### Reglerentwurf nach Reinisch

Bei Reinisch ist das Ziel, durch einen Regler eine **$\mathrm{IT}_1$-offene Kette** zu realisieren.
Große Zeitkonstanten werden gekürzt, um ein maximal schnell reagierendes System zu erhalten.
Stelle die Zeitkonstante $T_N$ so ein, dass du das gewünschte Verhalten bekommst.

In [4]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from IPython.display import display, Markdown

def get_plant_params(KS, T1, T2, T3):
    """Ermittelt die aktiven Zeitkonstanten und baut die Strecke GS(s) auf."""
    active_T = sorted([t for t in [T1, T2, T3] if t > 0], reverse=True)

    num_S = [KS]
    den_S = [1.0]
    for t_i in active_T:
        den_S = np.convolve(den_S, [t_i, 1.0])

    return active_T, num_S, den_S

def display_reinisch_formulas(show_controller, KS, T1, T2, T3, a):
    """Generiert passende LaTeX-Formeln je nach Streckenordnung (PT1, PT2, PT3)."""
    active_T, _, _ = get_plant_params(KS, T1, T2, T3)
    n = len(active_T)

    if n == 0:
        display(Markdown(f"$$\\mathbf{{Strecke}}\\ G_S(s): \\quad G_S(s) = {KS:.1f}$$"))
        return

    den_str = "".join([f"({t:.1f}s + 1)" for t in active_T])
    latex_GS = f"G_S(s) = \\frac{{{KS:.1f}}}{{{den_str}}}"

    if not show_controller:
        display(Markdown(f"$$\\mathbf{{Strecke}}\\ G_S(s): \\quad {latex_GS}$$"))
        return

    # Fallunterscheidung je nach Ordnung n
    if n == 1:
        # PT1 -> PI-Regler
        t1 = active_T[0]
        KR = 1.0 / (a * KS)
        latex_regler_typ = "PI-Regler"
        latex_GR = f"G_R(s) = {KR:.2f} \\cdot \\frac{{{t1:.1f}s + 1}}{{{t1:.1f}s}}"
        latex_G0 = f"G_0(s) = \\mathbf{{\\frac{{1}}{{{a:.1f} \\cdot {t1:.1f}s}}}}"

    elif n == 2:
        # PT2 -> PI-Regler (Kürzung von T1, T2 bildet IT1)
        t1, t2 = active_T[0], active_T[1]
        KR = t1 / (a * t2 * KS)
        latex_regler_typ = "PI-Regler"
        latex_GR = f"G_R(s) = {KR:.2f} \\cdot \\frac{{{t1:.1f}s + 1}}{{{t1:.1f}s}}"
        latex_G0 = f"G_0(s) = \\mathbf{{\\frac{{1}}{{{a:.1f} \\cdot {t2:.2f}s ({t2:.2f}s + 1)}}}}"

    else:  # n == 3
        # PT3 -> PID-Regler (Kürzung von T1 und T2, T3 bildet IT1)
        t1, t2, t3 = active_T[0], active_T[1], active_T[2]
        KR = t1 / (a * t3 * KS)
        latex_regler_typ = "PID-Regler"
        latex_GR = f"G_R(s) = {KR:.2f} \\cdot \\frac{{({t1:.1f}s + 1)({t2:.1f}s + 1)}}{{{t1:.1f}s}}"
        latex_G0 = f"G_0(s) = \\mathbf{{\\frac{{1}}{{{a:.1f} \\cdot {t3:.2f}s ({t3:.2f}s + 1)}}}}"

    latex_output = f"""
$$
\\begin{{aligned}}
\\mathbf{{Strecke\\ (PT_{{{n}}}):}} \\quad & {latex_GS} \\\\[6pt]
\\mathbf{{{latex_regler_typ}:}} \\quad & {latex_GR} \\quad (K_R = {KR:.2f}) \\\\[6pt]
\\mathbf{{Offene\\ Kette\\ G_0(s):}} \\quad & {latex_G0}
\\end{{aligned}}
$$
"""
    display(Markdown(latex_output))

def plot_step_response_reinisch(show_controller=False, KS=2.0, T1=4.5, T2=2.1, T3=0.8, a=2.0):
    display_reinisch_formulas(show_controller, KS, T1, T2, T3, a)

    active_T, num_S, den_S = get_plant_params(KS, T1, T2, T3)
    n = len(active_T)
    t = np.linspace(0, 15, 1000)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.axhline(1.0, color='black', ls='--', lw=1.2)

    if not show_controller or n == 0:
        sys_S = signal.TransferFunction(num_S, den_S)
        t_out, y_S = signal.step(sys_S, T=t)
        ax.plot(t_out, y_S, 'b-', lw=2.5, label='Ungeregelte Strecke')
    else:
        if n == 1:
            t1 = active_T[0]
            KR = 1.0 / (a * KS)
            num_R = [KR * t1, KR]
            den_R = [t1, 0.0]

            num_ideal = [1.0]
            den_ideal = [a * t1, 1.0]

        elif n == 2:
            t1, t2 = active_T[0], active_T[1]
            KR = t1 / (a * t2 * KS)
            num_R = [KR * t1, KR]
            den_R = [t1, 0.0]

            num_ideal = [1.0]
            den_ideal = [a * (t2**2), a * t2, 1.0]

        else:  # n == 3 (PID-Regler)
            t1, t2, t3 = active_T[0], active_T[1], active_T[2]
            KR = t1 / (a * t3 * KS)
            # Regler: K_R * (T1*s + 1)*(T2*s + 1) / (T1*s)
            num_R = [KR * t1 * t2, KR * (t1 + t2), KR]
            den_R = [t1, 0.0]

            num_ideal = [1.0]
            den_ideal = [a * (t3**2), a * t3, 1.0]

        # Reale Übertragungsfunktion des geschlossenen Regelkreises
        num_0 = np.convolve(num_R, num_S)
        den_0 = np.convolve(den_R, den_S)

        den_w = np.copy(den_0)
        den_w[-len(num_0):] += num_0

        sys_closed = signal.TransferFunction(num_0, den_w)
        t_out, y_closed = signal.step(sys_closed, T=t)

        sys_ideal = signal.TransferFunction(num_ideal, den_ideal)
        _, y_ideal = signal.step(sys_ideal, T=t)

        ax.plot(t_out, y_closed, 'r-', lw=2.5, label='Ausgang (mit Regler)')
        ax.plot(t_out, y_ideal, 'g--', lw=1.8, label='Soll-Verhalten')

    ax.set_title("Sprungantwort im Zeitbereich", fontsize=12)
    ax.set_xlabel("Zeit", fontsize=10)
    ax.set_ylabel("Amplitude", fontsize=10)
    ax.grid(True, ls="-", alpha=0.5)
    ax.legend(fontsize=10, loc="lower right")
    ax.set_ylim(-0.1, 2.0)
    ax.set_xlim(0, 15)

    plt.show()

# --- Interaktive Controls ---
w_show = widgets.Checkbox(value=False, description='Regler zuschalten')
w_KS   = widgets.FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1, description='K_S:')
w_a    = widgets.FloatSlider(value=2.0, min=0.5, max=3.0, step=0.1, description='Faktor a:')

w_T1   = widgets.FloatSlider(value=4.5, min=0.1, max=5.0, step=0.1, description='T1 [s]:')
w_T2   = widgets.FloatSlider(value=2.1, min=0.0, max=3.0, step=0.1, description='T2 [s]:')
w_T3   = widgets.FloatSlider(value=0.8, min=0.0, max=3.0, step=0.1, description='T3 [s]:')

interactive_plot = widgets.interactive_output(
    plot_step_response_reinisch,
    {'show_controller': w_show, 'KS': w_KS, 'T1': w_T1, 'T2': w_T2, 'T3': w_T3, 'a': w_a}
)

controls = widgets.VBox([
    w_show,
    widgets.HBox([w_KS, w_a]),
    widgets.HBox([w_T1, w_T2, w_T3])
])

display(interactive_plot, controls)

Output()

### Summenzeitkonstante
Mithilfe der Summenzeitkonstante ist es möglich die Ordnung höherer Systeme zu reduzieren. Dabei werden die kleineren Zeitkonstanten addiert.
 Die Flächen über der Sprungantwort für PT3, sowohl als auch für die Summenzeitkonstante sind gleich. Über die Fläche lässt sich, bei einem unbekannten System, ebenfalls die Summenzeitkonstante berechnen.



In [12]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import ipywidgets as widgets
from IPython.display import display, Markdown

# Feste Parameter
K_FIXED = 1.0
T_MAX_FIXED = 50.0  # Fester Zeitbereich für die X-Achse (0 bis 50 Sekunden)

def display_formulas(T1, T2, T3, T_sum):
    """Generiert die dynamischen LaTeX-Formeln für beide Übertragungsfunktionen."""
    active_T = [t for t in [T1, T2, T3] if t > 0]

    # Nenner des Originalsystems aufbauen
    den_terms = [f"({t:.1f}s + 1)" for t in active_T]
    den_str = "".join(den_terms) if den_terms else "1"

    latex_G_orig = f"G_{{orig}}(s) = \\frac{{{K_FIXED:.1f}}}{{{den_str}}}"
    latex_G_approx = f"G_{{approx}}(s) = \\frac{{{K_FIXED:.1f}}}{{{T_sum:.2f}s + 1}}"

    latex_output = f"""
$$
\\begin{{aligned}}
\\mathbf{{PT_3:}} \\quad & {latex_G_orig} \\\\[6pt]
\\mathbf{{PT_1:}} \\quad & {latex_G_approx}
\\end{{aligned}}
$$
"""
    display(Markdown(latex_output))

def plot_summenzeitkonstante(T1=2.0, T2=1.0, T3=0.5):
    T_sum = T1 + T2 + T3

    # Formeln über dem Diagramm ausgeben
    display_formulas(T1, T2, T3, T_sum)

    # Fester Zeitvektor
    t = np.linspace(0, T_MAX_FIXED, 1000)

    # 1. Originales PT3-System: G(s) = K / ((T1*s + 1)(T2*s + 1)(T3*s + 1))
    den1 = [T1, 1.0] if T1 > 0 else [1.0]
    den2 = [T2, 1.0] if T2 > 0 else [1.0]
    den3 = [T3, 1.0] if T3 > 0 else [1.0]

    den_orig = np.poly1d(den1) * np.poly1d(den2) * np.poly1d(den3)

    sys_orig = signal.TransferFunction([K_FIXED], den_orig.coefficients)
    _, y_orig = signal.step(sys_orig, T=t)

    # 2. Ersatz-PT1-System mit T_sigma
    sys_approx = signal.TransferFunction([K_FIXED], [T_sum, 1.0])
    _, y_approx = signal.step(sys_approx, T=t)

    # Diagramm erstellen
    fig, ax = plt.subplots(figsize=(9, 4.5))

    # Stationärer Endwert K = 1
    ax.axhline(K_FIXED, color='black', linestyle='--', linewidth=1.2, label=r'_nolegend_')

    # Kurven zeichnen
    ax.plot(t, y_orig, 'b-', linewidth=2.5, label='Originalsystem')
    ax.plot(t, y_approx, 'r--', linewidth=2.0, label='PT1-Ersatzmodell')

    # Flächen dauerhaft einfärben
    ax.fill_between(t, y_orig, K_FIXED, color='blue', alpha=0.15, label=f'_nolegend_')
    ax.fill_between(t, y_approx, K_FIXED, color='red', alpha=0.15, label=f'_nolegend_')

    # Plot-Formatierung
    ax.set_title("Summenzeitkonstante und Flächengleichheit", fontsize=12)
    ax.set_xlabel("Zeit", fontsize=10)
    ax.set_ylabel("Amplitude", fontsize=10)
    ax.grid(True, linestyle='-', alpha=0.5)
    ax.legend(fontsize=9, loc="lower right")

    # Feste Achsengrenzen
    ax.set_xlim(0, T_MAX_FIXED)
    ax.set_ylim(-0.05, 1.25)

    plt.show()

# Interaktive Schieberegler für T1, T2 und T3
w_T1 = widgets.FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1, description='T1 [s]:')
w_T2 = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='T2 [s]:')
w_T3 = widgets.FloatSlider(value=0.5, min=0.0, max=5.0, step=0.1, description='T3 [s]:')

interactive_widget = widgets.interactive_output(
    plot_summenzeitkonstante,
    {'T1': w_T1, 'T2': w_T2, 'T3': w_T3}
)

controls = widgets.HBox([w_T1, w_T2, w_T3])

display(interactive_widget, controls)

Output()

### Betragsoptimum
Das Ziel des Betragsoptimums ist es, den Betrag der Führungsübertragungsfunktion so lang und nah wie möglich 1 (0 dB) werden zu lassen.
Warum ist es nicht möglich für höhere Frequenzen dieses Verhalten zu erreichen?



In [16]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import ipywidgets as widgets
from IPython.display import display, Markdown
from matplotlib.lines import Line2D

# Deaktiviert MathText für Achsenbeschriftungen (verhindert den pyparsing-Fehler)
plt.rcParams['axes.formatter.use_mathtext'] = False

# Feste Streckenparameter
KS_FIXED = 1.0  # Streckenverstärkung
T1_FIXED = 2.0  # Große Zeitkonstante (wird durch TN = T1 kompensiert)

# Feste Achsengrenzen
W_MIN, W_MAX = 0.1, 50.0   # Kreisfrequenzbereich [rad/s]
T_MAX_FIXED = 10.0         # Zeitbereich [s]

def display_formulas(KR, T2):
    """Generiert dynamisch die Übertragungsfunktionen mit aktuellen Zahlenwerten."""
    coeff_s2 = (T1_FIXED * T2) / (KR * KS_FIXED)
    coeff_s1 = T1_FIXED / (KR * KS_FIXED)
    KR_BO = T1_FIXED / (2.0 * KS_FIXED * T2)

    latex_text = f"""
$$
\\begin{{aligned}}
\\mathbf{{Strecke}}\\ G_S(s): \\quad & \\frac{{{KS_FIXED:.1f}}}{{({T1_FIXED:.1f}s + 1)({T2:.2f}s + 1)}} \\\\[6pt]
\\mathbf{{PI-Regler}}\\ G_R(s): \\quad & {KR:.2f} \\cdot \\frac{{{T1_FIXED:.1f}s + 1}}{{{T1_FIXED:.1f}s}} \\\\[6pt]
\\mathbf{{Offene\\ Kette}}\\ G_0(s): \\quad & \\frac{{{KR * KS_FIXED:.2f}}}{{{T1_FIXED * T2:.2f} s^2 + {T1_FIXED:.1f} s}} = \\frac{{1}}{{{coeff_s2:.3f} s^2 + {coeff_s1:.2f} s}} \\\\[6pt]
\\mathbf{{Geschlossener\\ Regelkreis}}\\ G_w(s): \\quad & \\mathbf{{\\frac{{1}}{{{coeff_s2:.3f} s^2 + {coeff_s1:.2f} s + 1}}}} \\\\[6pt]
\\mathbf{{Betragsoptimum\\ für}}\\ K_R: \\quad & K_{{R,\\text{{BO}}}} = \\frac{{T_1}}{{2 \\cdot K_S \\cdot T_2}} = {KR_BO:.2f}
\\end{{aligned}}
$$
"""
    display(Markdown(latex_text))

def plot_pi_pt2_betragsoptimum(KR=2.0, T2=0.5):
    # 1. Kompensation der dominanten Zeitkonstante
    TN = T1_FIXED                       # Pol-Nullstellen-Kompensation
    KR_BO = TN / (2.0 * KS_FIXED * T2)  # Sollwert nach Betragsoptimum

    # Dynamic Markdown Output
    display_formulas(KR, T2)

    # 2. Übertragungsfunktionen definieren
    num_S = [KS_FIXED]
    den_S = [T1_FIXED * T2, T1_FIXED + T2, 1.0]

    num_R = [KR * TN, KR]
    den_R = [TN, 0.0]

    num_0 = np.polymul(num_R, num_S)
    den_0 = np.polymul(den_R, den_S)

    den_w = np.polyadd(den_0, num_0)
    sys_w = signal.TransferFunction(num_0, den_w)

    # 3. Frequenzgang berechnen (in dB)
    omega = np.logspace(np.log10(W_MIN), np.log10(W_MAX), 1000)
    _, mag_complex = signal.freqs(num_0, den_w, worN=omega)
    mag_db = 20 * np.log10(np.abs(mag_complex))

    # 4. Sprungantwort berechnen
    t = np.linspace(0, T_MAX_FIXED, 1000)
    _, y_t = signal.step(sys_w, T=t)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

    # Farbmarkierung: Grün bei Betragsoptimum (KR == KR_BO), Rot bei KR > KR_BO, Blau bei KR < KR_BO
    if abs(KR - KR_BO) < 1e-2:
        line_color = 'green'
        label_state = f'Betragsoptimum ($K_R = {KR:.2f}$)'
    elif KR > KR_BO:
        line_color = 'red'
        label_state = f'Überkompensiert ($K_R = {KR:.2f} > K_{{R,BO}}$)'
    else:
        line_color = 'blue'
        label_state = f'Unterkompensiert ($K_R = {KR:.2f} < K_{{R,BO}}$)'

    # --- Subplot 1: Betragsverlauf im Frequenzbereich (in dB) ---
    ax1.axhline(0.0, color='black', linestyle='--', linewidth=1.2, label=r'_nolegend_')
    ax1.semilogx(omega, mag_db, color=line_color, linewidth=2.5, label=label_state)

    ax1.set_title("Betragsverlauf im Frequenzbereich", fontsize=12)
    ax1.set_xlabel("Kreisfrequenz", fontsize=10)
    ax1.set_ylabel("Betrag", fontsize=10)
    ax1.grid(True, which='both', linestyle='-', alpha=0.5)

    # --- Subplot 2: Sprungantwort im Zeitbereich ---
    ax2.axhline(1.0, color='black', linestyle='--', linewidth=1.2, label=r'_nolegend_')
    ax2.plot(t, y_t, color=line_color, linewidth=2.5, label=label_state)

    ax2.set_title("Sprungantwort im Zeitbereich", fontsize=12)
    ax2.set_xlabel("Zeit", fontsize=10)
    ax2.set_ylabel("Amplitude", fontsize=10)
    ax2.grid(True, linestyle='-', alpha=0.5)

    # --- Legende ausschließlich im linken Subplot (ax1) ---
    legend_elements = [
        Line2D([0], [0], color='green', lw=2.5, label='Betragsoptimum'),
        Line2D([0], [0], color='red', lw=2.5, label='Zu groß, mehr Überschwingen'),
        Line2D([0], [0], color='blue', lw=2.5, label='Zu klein, trägeres Verhalten')
    ]

    ax1.legend(handles=legend_elements, fontsize=8, loc="lower left")

    # FESTE ACHSEN
    ax1.set_xlim(W_MIN, W_MAX)
    ax1.set_ylim(-20.0, 5.0)

    ax2.set_xlim(0, T_MAX_FIXED)
    ax2.set_ylim(-0.05, 1.6)

    plt.tight_layout()
    plt.show()

# Interaktive Schieberegler für KR und T2 (=T)
w_KR = widgets.FloatSlider(value=2.0, min=0.2, max=10.0, step=0.1, description='K_R:')
w_T2 = widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description='T2 (=T) [s]:')

interactive_widget = widgets.interactive_output(
    plot_pi_pt2_betragsoptimum,
    {'KR': w_KR, 'T2': w_T2}
)

controls = widgets.HBox([w_KR, w_T2])

display(interactive_widget, controls)

Output()

### Algebraischer Reglerentwurf
* Gibt es exakte Vorgaben für das Verhalten eines Reglers, so kann man mithilfe von Überschwingweite, Überschwingzeit
  und Beruhigungszeit ein Gebiet für zulässige Polstellen eingrenzen.
* Sind die Pole festgeleget, so lässt sich der Regler berechnen


In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['axes.formatter.use_mathtext'] = False

# Streckenparameter (PT1-Glied)
KS_FIXED = 1.0
T1_FIXED = 2.0

out = widgets.Output()

def h_to_phi(h_pct):
    """Rechnet Überschwingweite in % in den Dämpfungswinkel phi (in rad) um."""
    h = max(h_pct / 100.0, 1e-4)
    ln_h = np.log(h)
    D = -ln_h / np.sqrt(np.pi**2 + ln_h**2)
    return np.arccos(D), D

def plot_full_pole_region_from_specs(h_min, h_max, Tm_min, Tm_max, T5_min, T5_max, d_pole, w_pole):
    # 1. Umrechnungen von Gütekennwerten in s-Ebenen-Grenzen
    d_min = 3.0 / T5_max
    d_max = 3.0 / T5_min

    w_min = np.pi / Tm_max
    w_max = np.pi / Tm_min

    phi_min_rad, _ = h_to_phi(h_min)
    phi_max_rad, _ = h_to_phi(h_max)
    phi_min_deg = np.degrees(phi_min_rad)
    phi_max_deg = np.degrees(phi_max_rad)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2))

    # Grid für s-Ebene
    delta_axis_max = max(d_max * 1.4, 4.5)
    omega_axis_max = max(w_max * 1.4, 5.5)

    d_grid = np.linspace(0.01, delta_axis_max, 300)
    w_grid = np.linspace(0.01, omega_axis_max, 300)
    D_mesh, W_mesh = np.meshgrid(d_grid, w_grid)
    PHI_mesh = np.degrees(np.arctan2(W_mesh, D_mesh))

    # Maske für das dynamisch berechnete Polgebiet
    mask = (
        (D_mesh >= d_min) & (D_mesh <= d_max) &
        (W_mesh >= w_min) & (W_mesh <= w_max) &
        (PHI_mesh >= phi_min_deg) & (PHI_mesh <= phi_max_deg)
    )

    # Schattieren des Zielgebiets
    ax1.contourf(-D_mesh, W_mesh, np.where(mask, 1, 0), levels=[0.5, 1.5], colors=['#b0c4de'], alpha=0.7)
    ax1.contourf(-D_mesh, -W_mesh, np.where(mask, 1, 0), levels=[0.5, 1.5], colors=['#b0c4de'], alpha=0.7)

    # Grenzen zeichnen
    ax1.axvline(-d_min, color='red', linestyle='--', linewidth=1.2, label=r'$-\delta_{e,\mathrm{min}}$')
    ax1.axvline(-d_max, color='darkred', linestyle='--', linewidth=1.2, label=r'$-\delta_{e,\mathrm{max}}$')

    ax1.axhline(w_min, color='purple', linestyle=':', linewidth=1.2, label=r'$\omega_{e,\mathrm{min}}$')
    ax1.axhline(w_max, color='indigo', linestyle=':', linewidth=1.2, label=r'$\omega_{e,\mathrm{max}}$')
    ax1.axhline(-w_min, color='purple', linestyle=':', linewidth=1.2)
    ax1.axhline(-w_max, color='indigo', linestyle=':', linewidth=1.2)

    d_ray = np.array([0, delta_axis_max])
    ax1.plot(-d_ray, d_ray * np.tan(phi_min_rad), color='darkorange', linestyle='-.', linewidth=1.2, label=r'$\varphi_{\mathrm{min}}$')
    ax1.plot(-d_ray, d_ray * np.tan(phi_max_rad), color='orange', linestyle='-.', linewidth=1.2, label=r'$\varphi_{\mathrm{max}}$')
    ax1.plot(-d_ray, -d_ray * np.tan(phi_min_rad), color='darkorange', linestyle='-.', linewidth=1.2)
    ax1.plot(-d_ray, -d_ray * np.tan(phi_max_rad), color='orange', linestyle='-.', linewidth=1.2)

    ax1.axhline(0, color='black', linewidth=1.0)
    ax1.axvline(0, color='black', linewidth=1.0)

    # Manuell einstellbares Polpaar zeichnen (ohne "Zentrumspol" in der Legende)
    ax1.plot([-d_pole, -d_pole], [w_pole, -w_pole], 'x', color='green', markersize=9, markeredgewidth=2.5, label=r'Polpaar $s_{1,2}$')

    ax1.set_title(r"Zielgebiet in der $s$-Ebene", fontsize=11)
    ax1.set_xlabel(r"Realteil $\mathrm{Re}\{s\}$", fontsize=10)
    ax1.set_ylabel(r"Imaginärteil $\mathrm{Im}\{s\}$", fontsize=10)
    ax1.set_xlim(-delta_axis_max, 0.5)
    ax1.set_ylim(-omega_axis_max, omega_axis_max)
    ax1.grid(True, linestyle='-', alpha=0.3)
    ax1.legend(loc="upper left", fontsize=8, ncol=2)

    # --- Subplot 2: Sprungantwort des eingestellten Pols ---
    omega0 = np.sqrt(d_pole**2 + w_pole**2)
    T_val = 1.0 / omega0
    D_val = d_pole / omega0

    TN = (2 * D_val * T_val * T1_FIXED - T_val**2) / T1_FIXED
    KP = (2 * D_val * T1_FIXED - T_val) / (KS_FIXED * T_val)

    num_closed = [KP * KS_FIXED * TN, KP * KS_FIXED]
    den_closed = [TN * T1_FIXED, TN + KP * KS_FIXED * TN, KP * KS_FIXED]
    sys_w = signal.TransferFunction(num_closed, den_closed)

    t_max = max(T5_max * 1.3, 6.0)
    t = np.linspace(0, t_max, 1000)
    _, y_t = signal.step(sys_w, T=t)

    ax2.plot(t, y_t, color='green', linewidth=2.0, label='Sprungantwort')
    ax2.axhline(1.0, color='black', linestyle='--', linewidth=1.0)

    # Schranken
    ax2.axhline(1.0 + h_max/100.0, color='orange', linestyle=':', label=r'$\Delta h_{\mathrm{max}}$')
    ax2.axhline(1.0 + h_min/100.0, color='darkorange', linestyle=':', label=r'$\Delta h_{\mathrm{min}}$')

    ax2.set_title(r"Sprungantwort $y(t)$", fontsize=11)
    ax2.set_xlabel("Zeit $t$ [s]", fontsize=10)
    ax2.set_ylabel(r"Regelgröße $y(t)$", fontsize=10)
    ax2.set_xlim(0, t_max)
    ax2.set_ylim(-0.05, max(1.0 + h_max/100.0 * 1.3, 1.4))
    ax2.grid(True, linestyle='-', alpha=0.4)
    ax2.legend(loc="lower right", fontsize=8)

    plt.tight_layout()
    plt.show()

def update_plot(change=None):
    with out:
        clear_output(wait=True)
        h_min = min(w_hmin.value, w_hmax.value - 1.0)
        h_max = max(w_hmax.value, h_min + 1.0)

        Tm_min = min(w_Tmmin.value, w_Tmmax.value - 0.1)
        Tm_max = max(w_Tmmax.value, Tm_min + 0.1)

        T5_min = min(w_T5min.value, w_T5max.value - 0.2)
        T5_max = max(w_T5max.value, T5_min + 0.2)

        d_pole = w_dpole.value
        w_pole = w_wpole.value

        plot_full_pole_region_from_specs(h_min, h_max, Tm_min, Tm_max, T5_min, T5_max, d_pole, w_pole)

# Gütekennwert-Slider
w_hmin = widgets.FloatSlider(value=5.0, min=1.0, max=30.0, step=1.0, description='h_min [%]:')
w_hmax = widgets.FloatSlider(value=25.0, min=5.0, max=50.0, step=1.0, description='h_max [%]:')

w_Tmmin = widgets.FloatSlider(value=0.8, min=0.2, max=3.0, step=0.1, description='T_m,min [s]:')
w_Tmmax = widgets.FloatSlider(value=2.5, min=0.5, max=5.0, step=0.1, description='T_m,max [s]:')

w_T5min = widgets.FloatSlider(value=1.0, min=0.5, max=4.0, step=0.1, description='T_5%,min [s]:')
w_T5max = widgets.FloatSlider(value=4.0, min=1.5, max=8.0, step=0.1, description='T_5%,max [s]:')

# Regler zur Polverschiebung: Re{s} = -delta, Im{s} = omega
w_dpole = widgets.FloatSlider(value=1.25, min=0.1, max=5.0, step=0.05, description='Re{s} (-δ):')
w_wpole = widgets.FloatSlider(value=2.0, min=0.1, max=6.0, step=0.05, description='Im{s} (ω):')

all_widgets = [w_hmin, w_hmax, w_Tmmin, w_Tmmax, w_T5min, w_T5max, w_dpole, w_wpole]
for w in all_widgets:
    w.observe(update_plot, names='value')

update_plot()

box1 = widgets.VBox([w_hmin, w_hmax])
box2 = widgets.VBox([w_Tmmin, w_Tmmax])
box3 = widgets.VBox([w_T5min, w_T5max])
box4 = widgets.VBox([widgets.HTML("<b>Polposition (s₁;₂ = -δ ± jω)</b>"), w_dpole, w_wpole])

controls = widgets.VBox([
    widgets.HBox([box1, box2, box3]),
    box4
])

display(controls, out)

Output()